# Prop-DeOccNet Training — Kaggle

**Sebelum mulai:**
1. Aktifkan GPU: *Settings → Accelerator → GPU T4 x2* (atau P100)
2. Aktifkan Internet: *Settings → Internet → On*
3. Tambahkan dataset Roboflow COCO: *+ Add Input → cari dataset kamu*

**Dataset yang perlu ditambahkan:**
- **Dataset anotasi** (upload dulu dari lokal): berisi folder `train/`, `valid/`, `test/` dengan `_annotations.coco.json`
- *(Opsional)* **Dataset checkpoint** dari run sebelumnya untuk melanjutkan training

---

## Step 1 — Cek GPU

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    for i in range(n):
        name = torch.cuda.get_device_name(i)
        vram = round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
        print(f'  GPU {i}: {name} ({vram} GB)')
else:
    print('WARNING: GPU tidak aktif — aktifkan di Settings → Accelerator')

## Step 2 — Konfigurasi

Edit variabel di cell ini sesuai setup kamu.

### Cara mendapatkan nama dataset Kaggle
Setelah menambahkan dataset via *+ Add Input*, lihat path yang muncul di panel kiri.
Contoh: `/kaggle/input/daun-itoh-coco/` → `DATASET_NAME = "daun-itoh-coco"`

### Cara upload dataset Roboflow ke Kaggle
1. Zip folder `annotation-v1-roboflow.v2i.coco` → `daun-itoh-coco.zip`
2. Buka [kaggle.com/datasets](https://www.kaggle.com/datasets) → *New Dataset*
3. Upload zip, beri nama `daun-itoh-coco`
4. Tambahkan ke notebook ini via *+ Add Input*

In [ ]:
# ── Konfigurasi (edit sesuai setup kamu) ──────────────────────────────────────

# Nama dataset Kaggle yang berisi folder train/valid/test + _annotations.coco.json
# Cek path di panel kiri: /kaggle/input/<DATASET_NAME>/
DATASET_NAME = "daun-itoh-coco"  # <-- ganti ini

# URL repo GitHub (isi jika ingin clone kode dari git, kosongkan jika upload kode sebagai dataset)
REPO_URL = ""  # contoh: "https://github.com/username/labeling-daun-itoh.git"

# Nama dataset Kaggle yang berisi kode project (hanya dipakai kalau REPO_URL kosong)
CODE_DATASET_NAME = "labeling-daun-itoh-code"  # <-- ganti ini jika REPO_URL kosong

# Nama dataset Kaggle berisi checkpoint dari run sebelumnya (kosong = mulai dari awal)
CHECKPOINT_DATASET_NAME = ""  # contoh: "labeling-daun-itoh-checkpoints"

# Hyperparameter training (override default di config_kaggle.yaml)
EPOCHS      = 50
BATCH_SIZE  = 4    # T4 16GB: 4; P100 16GB: 6; jika OOM, turunkan ke 2
IMAGE_SIZE  = 512
BACKBONE    = "resnet101"  # resnet50 (lebih cepat) | resnet101 (lebih akurat)

# ── Path turunan (tidak perlu diedit) ─────────────────────────────────────────
DATASET_PATH   = f"/kaggle/input/{DATASET_NAME}"
WORK_DIR       = "/kaggle/working/labeling-daun-itoh"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
RUNS_DIR       = "/kaggle/working/runs"
CONFIG_PATH    = f"{WORK_DIR}/training/config_kaggle.yaml"

print(f"Dataset path : {DATASET_PATH}")
print(f"Working dir  : {WORK_DIR}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")

## Step 3 — Setup Kode Project

Notebook ini akan clone kode dari GitHub (jika `REPO_URL` diisi) atau copy dari Kaggle dataset.

**Cara upload kode sebagai Kaggle dataset (jika `REPO_URL` kosong):**
1. Zip isi folder project (termasuk folder `training/`)
2. Upload ke Kaggle → beri nama sesuai `CODE_DATASET_NAME`
3. Tambahkan ke notebook via *+ Add Input*

In [ ]:
import os, sys, shutil
from pathlib import Path

if REPO_URL:
    # Clone dari GitHub
    if not Path(WORK_DIR).exists():
        ret = os.system(f"git clone {REPO_URL} {WORK_DIR}")
        if ret != 0:
            raise RuntimeError("git clone gagal — pastikan Internet aktif dan URL benar")
    else:
        print(f"Sudah ada: {WORK_DIR}")
    print(f"Kode di-clone ke: {WORK_DIR}")
else:
    # Copy dari Kaggle dataset
    CODE_SRC = f"/kaggle/input/{CODE_DATASET_NAME}"
    if not Path(CODE_SRC).exists():
        raise FileNotFoundError(
            f"Dataset kode tidak ditemukan: {CODE_SRC}\n"
            "Tambahkan dataset via '+ Add Input' atau isi REPO_URL dengan URL GitHub."
        )
    if not Path(WORK_DIR).exists():
        shutil.copytree(CODE_SRC, WORK_DIR)
    print(f"Kode di-copy dari: {CODE_SRC}")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"Working dir  : {os.getcwd()}")
print("Isi folder   :", sorted(os.listdir('.')))

## Step 4 — Install Dependencies

Kaggle sudah menyediakan PyTorch, NumPy, OpenCV, albumentations, dan tqdm.
Yang perlu diinstall hanya **pycocotools** (di-build dari source agar cocok dengan NumPy versi Kaggle).

> Setelah cell ini selesai: ***Session → Restart & Run All*** kalau muncul error NumPy, lalu lanjut dari Step 5.

In [ ]:
import subprocess, sys

# Build pycocotools dari source agar cocok dengan numpy versi Kaggle
print("Installing pycocotools...")
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "cython>=3.0.0", "-q"],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "pycocotools", "--no-binary", "pycocotools", "--no-cache-dir", "-q"],
    check=True
)

# Verifikasi
import numpy as np
import torch
import pycocotools._mask  # error di sini berarti build gagal
import albumentations as A
print(f"numpy        : {np.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
print("pycocotools  : OK")
print(f"albumentations: {A.__version__}")
print("\nDependencies OK!")

## Step 5 — Tulis Config & Verifikasi Dataset

In [ ]:
import yaml, json
from pathlib import Path

# Tulis config_kaggle.yaml dengan path yang sesuai
config = {
    # Data
    "train_json": f"{DATASET_PATH}/train/_annotations.coco.json",
    "val_json":   f"{DATASET_PATH}/valid/_annotations.coco.json",
    "test_json":  f"{DATASET_PATH}/test/_annotations.coco.json",
    "images_dir": None,
    "num_classes": 3,
    # Model
    "backbone": BACKBONE,
    "pretrained_backbone": True,
    "aspp_rates": [6, 12, 18, 24],
    "aspp_out_channels": 256,
    "trainable_backbone_layers": 3,
    "use_boundary_head": True,
    # Training
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "num_workers": 2,
    "image_size": IMAGE_SIZE,
    "mosaic_prob": 0.3,
    # Normalization
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std":  [0.229, 0.224, 0.225],
    # Optimizer
    "optimizer": "adam",
    "lr": 1e-4,
    "weight_decay": 1e-4,
    # Scheduler
    "lr_scheduler": "cosine",
    "lr_min": 1e-6,
    # Loss
    "loss_weights": {"focal": 1.0, "dice": 1.0, "boundary": 1.0},
    # Checkpoint
    "checkpoint_dir": CHECKPOINT_DIR,
    "save_every": 5,
    "resume": None,
    # Logging
    "tensorboard_dir": RUNS_DIR,
    "log_every": 20,
    # Oklusi
    "occlusion_thresholds": [0.15, 0.40],
    "occlusion_filter": None,
    "occlusion_stats": True,
}

Path(CONFIG_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(CONFIG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
print(f"Config ditulis ke: {CONFIG_PATH}")

# Verifikasi dataset
print("\nVerifikasi dataset:")
for split in ["train", "valid", "test"]:
    ann_path = Path(DATASET_PATH) / split / "_annotations.coco.json"
    if ann_path.exists():
        with open(ann_path) as f:
            data = json.load(f)
        n_img = len(data["images"])
        n_ann = len(data["annotations"])
        n_cat = len(data["categories"])
        img_dir = ann_path.parent
        n_files = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        print(f"  [{split:5s}] {n_img:4d} gambar | {n_ann:5d} anotasi | "
              f"{n_cat} kategori | {n_files} file gambar")
    else:
        print(f"  [{split:5s}] TIDAK DITEMUKAN: {ann_path}")

print("\nKategori:")
with open(f"{DATASET_PATH}/train/_annotations.coco.json") as f:
    cats = json.load(f)["categories"]
for c in cats:
    print(f"  id={c['id']} → {c['name']} (supercategory: {c.get('supercategory', '-')})")

## Step 6 — Statistik Oklusi

Menghitung jumlah gambar berdasarkan tingkat oklusi (rendah / sedang / tinggi).
Oklusi dihitung dari **overlap antar mask** karena dataset Roboflow tidak menyimpan field `occlusion` secara eksplisit.

| Level | Definisi |
|-------|----------|
| **Rendah** | < 15% area daun tertutup daun lain |
| **Sedang** | 15% – 40% area tertutup |
| **Tinggi** | ≥ 40% area tertutup |

In [ ]:
from training.dataset import compute_dataset_occlusion_stats

occlusion_thresholds = (0.15, 0.40)

all_stats = {}
for split in ["train", "valid", "test"]:
    ann_path = f"{DATASET_PATH}/{split}/_annotations.coco.json"
    print(f"\nOklusi [{split}]:")
    stats = compute_dataset_occlusion_stats(ann_path, thresholds=occlusion_thresholds, verbose=True)
    all_stats[split] = stats

# Ringkasan
print("\n" + "="*55)
print(f"{'Split':<8} {'Rendah':>8} {'Sedang':>8} {'Tinggi':>8} {'Total':>8}")
print("-"*55)
for split, s in all_stats.items():
    t = max(s['total'], 1)
    print(f"{split:<8} {s['rendah']:>5} ({s['rendah']/t:>4.0%})",
          f"{s['sedang']:>5} ({s['sedang']/t:>4.0%})",
          f"{s['tinggi']:>5} ({s['tinggi']/t:>4.0%})",
          f"{s['total']:>8}")
print("="*55)

## Step 7 — Load Checkpoint dari Run Sebelumnya (Opsional)

Untuk melanjutkan training dari checkpoint:
1. Download folder `checkpoints/` dari output run sebelumnya
2. Upload ke Kaggle sebagai dataset baru
3. Tambahkan ke notebook via *+ Add Input*
4. Set `CHECKPOINT_DATASET_NAME` di Step 2

Training akan otomatis melanjutkan dari epoch terakhir (auto-resume dari `epoch_XXX.pth` terbaru).

In [ ]:
import shutil
from pathlib import Path

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

if CHECKPOINT_DATASET_NAME:
    ckpt_src = Path(f"/kaggle/input/{CHECKPOINT_DATASET_NAME}")
    if not ckpt_src.exists():
        raise FileNotFoundError(
            f"Dataset checkpoint tidak ditemukan: {ckpt_src}\n"
            "Tambahkan dataset via '+ Add Input' atau kosongkan CHECKPOINT_DATASET_NAME."
        )
    pth_files = list(ckpt_src.glob("*.pth"))
    for f in pth_files:
        shutil.copy2(f, Path(CHECKPOINT_DIR) / f.name)
    print(f"Loaded {len(pth_files)} checkpoint(s) dari {ckpt_src}:")
    for f in sorted(Path(CHECKPOINT_DIR).glob("*.pth")):
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.0f} MB)")
else:
    print("Tidak ada checkpoint sebelumnya — training dimulai dari awal.")
    print("(Set CHECKPOINT_DATASET_NAME di Step 2 untuk melanjutkan training)")

## Step 8 — Training

Estimasi waktu di Kaggle GPU:
| GPU | batch_size | image_size | 50 epoch |
|-----|-----------|-----------|----------|
| T4 (16GB) | 4 | 512 | ~5–8 jam |
| P100 (16GB) | 6 | 512 | ~4–6 jam |
| T4 x2 | 4 | 512 | ~3–5 jam |

**Auto-resume:** Training otomatis lanjut dari `epoch_XXX.pth` terakhir di `CHECKPOINT_DIR`.

**Checkpoint disimpan di:** `/kaggle/working/checkpoints/` — download setelah training selesai.

In [ ]:
import sys
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

from training.train import train

train(
    config_path=CONFIG_PATH,
    # Tidak perlu drive_checkpoint_dir — checkpoint ada di /kaggle/working/
)

## Step 9 — TensorBoard

Pantau training secara live atau setelah selesai.
Pastikan cell training sudah berjalan minimal 1 epoch.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/runs/

## Step 10 — Evaluasi Model Terbaik

In [ ]:
import torch, yaml
from torch.utils.data import DataLoader
from training.model import PropDeOccNet
from training.dataset import DaunDataset, collate_fn, get_val_transforms
from training.evaluate import evaluate

cfg = yaml.safe_load(open(CONFIG_PATH))

best_ckpt_path = f"{CHECKPOINT_DIR}/best.pth"
if not Path(best_ckpt_path).exists():
    raise FileNotFoundError(f"best.pth tidak ditemukan di {CHECKPOINT_DIR} — training belum selesai?")

model = PropDeOccNet(
    num_classes=cfg["num_classes"],
    backbone=cfg["backbone"],
    aspp_rates=cfg["aspp_rates"],
    use_boundary_head=cfg["use_boundary_head"],
)
ckpt = torch.load(best_ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model = model.to("cuda")

print(f"Loaded best.pth dari epoch {ckpt['epoch'] + 1} (BF Score: {ckpt.get('bf_score', 0):.4f})")

test_ds = DaunDataset(
    coco_json_path=cfg["test_json"],
    images_dir=cfg.get("images_dir"),
    transforms=get_val_transforms(cfg["image_size"]),
)
test_loader = DataLoader(test_ds, batch_size=1, collate_fn=collate_fn, num_workers=2)

print(f"\nEvaluasi pada {len(test_ds)} gambar test ...")
metrics = evaluate(model, test_loader, device="cuda")

print("\n=== Test Set Results ===")
for k, v in metrics.items():
    print(f"  {k:<18}: {v:.4f}")

## Step 11 — Output & Download

Semua output tersimpan di `/kaggle/working/`.
Download melalui panel kanan → *Output* tab, atau via [Kaggle API](https://github.com/Kaggle/kaggle-api):
```bash
kaggle kernels output <username>/<notebook-name> -p /path/to/local/dir
```

**Untuk melanjutkan training di sesi berikutnya:**
1. Download folder `checkpoints/` dari Output tab
2. Upload ke Kaggle sebagai dataset baru (misal: `labeling-daun-itoh-checkpoints`)
3. Set `CHECKPOINT_DATASET_NAME` di Step 2 pada run berikutnya

In [ ]:
import os
from pathlib import Path

working = Path("/kaggle/working")

print("=== Output di /kaggle/working/ ===")

# Checkpoints
ckpt_dir = working / "checkpoints"
if ckpt_dir.exists():
    pth_files = sorted(ckpt_dir.glob("*.pth"))
    print(f"\ncheckpoints/ ({len(pth_files)} file):")
    for f in pth_files:
        print(f"  {f.name:<25} {f.stat().st_size / 1e6:>7.1f} MB")
else:
    print("  checkpoints/ — belum ada")

# TensorBoard runs
runs_dir = working / "runs"
if runs_dir.exists():
    run_dirs = [d for d in runs_dir.rglob("events.out.tfevents.*")]
    total_tb = sum(f.stat().st_size for f in run_dirs)
    print(f"\nruns/ ({len(run_dirs)} event file, total {total_tb/1e6:.1f} MB)")
else:
    print("  runs/ — belum ada")

print("\nDownload via Output tab di panel kanan →")